# Taxi Fare Prediction with a GAM

This notebook contains an entire pipeline to predict NYC taxi fare amount
using only non-fare-related trip features by fitting a Generalized Additive Model
(GAM). The notebook loads NYC TLC-style trip data, prepares relevant features,
fits a model, evaluates it, and finally contains some interpretations on the
smooth effects and on the model itself.

### Configuration

Use this code cell to configure the global variables needed for analysis.

`PATH`: Path to parquet file or directory.  
`MAX_ROWS`: Optional maximum number of rows to use for analysis (`None` for all)  
`SEED`: Random seed for reproducibility  
`SPLIT`: Train / validation split fraction (e.g. `0.8` for 80% training)  

In [4]:
PATH = "s3://dsc291-ucsd/taxi/Dataset/2009/yellow_taxi/yellow_tripdata_2009-01.parquet"
MAX_ROWS = None
SEED = 0
SPLIT = 0.8

## Loading and Preparing Data

The following cells load the parquet file(s) from the configured path. Because
NYC TLC data schema differs from year to year, column names are normalized
across varied schema naming conventions. We also want to only read in the
required columns to efficiently utilize memory and reduce computation time.
We can find the columns we need by looking at the defined schemas in the `schemas`
directory for each taxi type.

| Taxi Type | Pickup Datetime | Dropoff Datetime | Trip Distance | Fare Amount | 
|-----------|-----------------|------------------|---------------|-------------|
| FHV | pickup_datetime | dropOff_datetime | -- | -- |
| Green | lpep_pickup_datetime | lpep_dropoff_datetime | trip_distance | fare_amount |
| HVFHS | pickup_datetime | dropoff_datetime | trip_miles | base_passenger_fare | 
| Yellow | tpep_pickup_datetime | tpep_dropoff_datetime | trip_distance | fare_amount |

Note that `HVFHS` data also contains a `trip_time` data column, however we will
calculate this on our own to keep data processing consistent between taxi types.
Because `FHV` data doesn't contain any relevant trip fare data for our needs,
we will disregard any `FHV` type trips.

Datetimes are parsed to derive trip duration (minutes), as well as the hour 
and day of the week the trip occurred. Any rows with missing or invalid fare,
distance, or duration values are dropped. Finally, the data is randomly split
into the training and testing split with the random seed. 

Our base model does not factor in the actual datetime of the trips. Arguably,
taxi fares should increase across the years due to inflation. Several assumptions
are made for the data from the inputted path consequently:
- data is only about one type of taxi
- data is only from a single year of trips

In [5]:
from __future__ import annotations

from pathlib import Path
from typing import List

import fsspec

def is_s3_path(path: str | Path) -> bool:
    """
    Return True if the path is an S3 URI.
    """
    return str(path).startswith("s3://")

def get_filesystem(path: str | Path):
    """
    Return an fsspec filesystem for local or S3 paths.
    """
    if is_s3_path(path):
        return fsspec.filesystem("s3", anon=True)
    return fsspec.filesystem("file")

def discover_parquet_files(input_path: str | Path) -> List[str]:
    """
    Recursively discover all .parquet files from a local directory or S3 prefix.
    Returns a sorted list of fully-qualified paths.
    """
    input_path = str(input_path)

    if input_path.endswith(".parquet"):
        return [input_path]

    fs = get_filesystem(input_path)

    if is_s3_path(input_path):
        # Remove scheme for fs.walk
        stripped = input_path.replace("s3://", "", 1)
        base = stripped.rstrip("/")
        files = []

        for root, _, filenames in fs.walk(base):
            for name in filenames:
                if name.endswith(".parquet"):
                    files.append(f"s3://{root}/{name}")

    else:
        base = Path(input_path)
        if not base.exists():
            raise FileNotFoundError(f"Input path does not exist: {input_path}")

        files = [
            str(p)
            for p in base.rglob("*.parquet")
            if p.is_file()
        ]

    files.sort()

    if not files:
        raise FileNotFoundError("No Parquet files found under %s", input_path)

    return files

def find_pickup_datetime_col(columns: List[str]) -> str:
    """
    Detect exact pickup datetime column name.
    """
    candidates = [
        "Trip_Pickup_DateTime",
        "lpep_pickup_datetime",
        "pickup_datetime",
        "tpep_pickup_datetime",
    ]
    for cand in candidates:
        if cand in columns:
            return cand
    return None

def find_dropoff_datetime_col(columns: List[str]) -> str:
    """
    Detect exact dropoff datetime column name.
    """
    candidates = [
        "Trip_Dropoff_DateTime",
        "lpep_dropoff_datetime",
        "dropoff_datetime",
        "dropOff_datetime",
        "tpep_dropoff_datetime",
    ]
    for cand in candidates:
        if cand in columns:
            return cand
    return None

def find_fare_col(columns: List[str]) -> str:
    """
    Detect exact fare column name.
    """
    candidates = [
        "Fare_Amt",
        "base_passenger_fare",
        "fare_amount"
    ]
    for cand in candidates:
        if cand in columns:
            return cand
    return None

def infer_taxi_type_from_path(file_path: str | Path) -> str:
    """
    Infer taxi type (yellow, green, fhv, hvfhv) from file path.
    """
    path = str(file_path).lower()
    for taxi in ["yellow", "green", "hvfhv", "fhv"]:
        if taxi in path:
            return taxi
    return "unknown"

In [10]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.fs as fs

from concurrent.futures import ThreadPoolExecutor, as_completed

print("Reading data from:")
print(PATH)

files = discover_parquet_files(PATH)
print(f"Discovered {len(files)} Parquet files.")

def process_file(file: str, max_rows: int) -> pd.DataFrame:
    print(f"Reading {file}...")
    schema = pq.read_schema(file, filesystem=get_filesystem(file))
    taxi_type = infer_taxi_type_from_path(file)

    if taxi_type == "fhv":
        print(f"Skipping FHV file {file} since it lacks fare information.")
        return
    elif taxi_type == "unknown":
        print(f"Skipping file {file} since taxi type could not be inferred.")
        return

    pickup_dt_col = find_pickup_datetime_col(schema.names)
    dropoff_dt_col = find_dropoff_datetime_col(schema.names)
    fare_col = find_fare_col(schema.names)

    if not pickup_dt_col or not dropoff_dt_col or not fare_col:
        print(f"Skipping file {file} since required columns could not be detected.")
        return

    if is_s3_path(file):
        file_path = file[5:]

    pf = pq.ParquetFile(file_path, filesystem=fs.S3FileSystem(anonymous=True, region="us-west-2") if is_s3_path(file) else fs.LocalFileSystem())
    if max_rows is None:
        table = pf.read(columns=[pickup_dt_col, dropoff_dt_col, fare_col], use_threads=True)
    else:
        remaining = max_rows
        chunks: list[pa.Table] = []

        for batch in pf.iter_batches(columns=[pickup_dt_col, dropoff_dt_col, fare_col], use_threads=True):
            if remaining <= 0:
                break
            batch_size = batch.num_rows
            if batch_size > remaining:
                batch = batch.slice(0, remaining)
            chunks.append(batch)
            remaining -= batch_size
        table = pa.concat_tables(chunks)

    rename_map = {
        pickup_dt_col: "pickup_datetime",
        dropoff_dt_col: "dropoff_datetime",
        fare_col: "fare_amount",
    }

    table = table.rename_columns([rename_map.get(name, name) for name in table.column_names])
    df = table.to_pandas()

    return df

dfs = []
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(process_file, file, MAX_ROWS) for file in files]
    for future in as_completed(futures):
        df = future.result()
        if df is not None:
            dfs.append(df)

if not dfs:
    raise ValueError("No valid Parquet files found with required columns.")
print(f"Successfully read {len(dfs)} DataFrames. Concatenating...")

data = pd.concat(dfs, ignore_index=True, copy=False)
pd.set_option('display.max_columns', None)
data

Reading data from:
s3://dsc291-ucsd/taxi/Dataset/2009/yellow_taxi/yellow_tripdata_2009-01.parquet
Discovered 1 Parquet files.
Reading s3://dsc291-ucsd/taxi/Dataset/2009/yellow_taxi/yellow_tripdata_2009-01.parquet...
Successfully read 1 DataFrames. Concatenating...


,pickup_datetime,dropoff_datetime,fare_amount
0,2009-01-04 02:52:00,2009-01-04 03:02:00,8.9
1,2009-01-04 03:31:00,2009-01-04 03:38:00,12.1
2,2009-01-03 15:43:00,2009-01-03 15:57:00,23.7
3,2009-01-01 20:52:58,2009-01-01 21:14:00,14.9
4,2009-01-24 16:18:23,2009-01-24 16:24:56,3.7
...,...,...,...
14092408,2009-01-27 14:36:00,2009-01-27 14:46:00,6.5
14092409,2009-01-27 13:56:00,2009-01-27 14:02:00,8.1
14092410,2009-01-23 08:39:44,2009-01-23 09:02:15,14.5
14092411,2009-01-24 23:05:00,2009-01-24 23:15:00,10.9
